# Simple: Categorical Generalization with PAMOLA.CORE

**Goal**: Learn categorical generalization basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Apply frequency-based generalization using execute()
- Compare before/after results
- Understand privacy improvements

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- Verifies all imports work correctly

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/  ← Data directory
    └── anonymization/generalization/
        └── 01_...simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display

print("🔍 Detecting PAMOLA installation...\n")

# Get current notebook location
current_dir = Path.cwd()
print(f"📍 Notebook location: {current_dir}")

# Navigate up to find project root (where pamola_core exists)
project_root = current_dir

# Go up maximum 6 levels to find pamola_core
for level in range(6):
    if (project_root / 'pamola_core').exists():
        print(f"✓ Found pamola_core at level {level}: {project_root}")
        break
    project_root = project_root.parent
else:
    raise FileNotFoundError(
        f"❌ Could not find pamola_core directory!\n"
        f"Searched up to: {project_root}\n"
        f"Current location: {current_dir}"
    )

# Add to Python path
sys.path.insert(0, str(project_root))
print(f"✓ Added to sys.path\n")

# Import PAMOLA modules
print("📦 Importing PAMOLA modules...")
from pamola_core.anonymization.generalization.categorical_op import (
    CategoricalGeneralizationOperation
)
from pamola_core.utils.ops.op_data_source import DataSource
from pamola_core.utils.progress import HierarchicalProgressTracker
from pamola_core.utils.tasks.task_reporting import TaskReporter

print("✅ All imports successful!\n")
print("=" * 80)
print(f"📂 Project root:  {project_root.name}")
print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
print("=" * 80)

## Step 2: Load Sample Data

**How to use:**
- Load data from `examples/data_examples/category_hierarchy_data.csv`
- Auto-create sample data if file doesn't exist
- Inspect the dataset structure

**Expected output:** DataFrame with records including categories, locations, etc.

In [ ]:
# Define path to sample data (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'category_hierarchy_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

# Check if file exists
if not data_path.exists():
    print(f"⚠️  File not found: {data_path}")
    print("Creating sample data...\n")
    
    # Create directory
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data
    sample_data = pd.DataFrame({
        'record_id': range(1, 21),
        'category': [
            'Category A1', 'Category A', 'Category A2',
            'Category B1', 'Category B2', 'Category C',
            'Category C1', 'Category D', 'Category E1',
            'Category E2', 'Category A', 'Category F1',
            'Category F2', 'Category G', 'Category H',
            'Category A', 'Category I', 'Category I1',
            'Category J', 'Category K'
        ],
        'region': [
            'North', 'North', 'North', 'South', 'South',
            'East', 'East', 'West', 'West', 'West',
            'North', 'South', 'South', 'East', 'East',
            'North', 'North', 'North', 'Central', 'Central'
        ],
        'value': [
            120, 95, 70, 110, 135, 115,
            140, 105, 65, 85, 95, 75,
            55, 90, 95, 95, 100, 125,
            90, 60
        ]
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created at: {data_path}\n")

# Load data using pandas
df = pd.read_csv(data_path)

print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if df[col].dtype == 'object':
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
- Create task directory for outputs
- Initialize TaskReporter for tracking
- Setup DataSource with our DataFrame
- Configure progress tracker
- **Configure field name** for anonymization

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "category"  # Customize this!
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "category"  # Field to anonymize - CUSTOMIZE THIS!
    }

# Get config
kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists in dataset
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to anonymize: '{field_name}'")
print(f"  Unique values: {df[field_name].nunique()}")
print(f"  Sample values: {list(df[field_name].unique()[:5])}")
print(f"  Distribution:")
print(df[field_name].value_counts().head())

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="anonymization",
    description="Anonymization of 'category' field using categorical generalization.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config kwargs (without field_name for execute)
execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Create CategoricalGeneralizationOperation with full config
- Use `operation.execute()` with explicit execution configs
- Monitor progress through tracker

**Key parameters:**
- `strategy='frequency_based'`: Group rare categories
- `freq_threshold=0.1`: Categories < 10% → "OTHER"
- `max_categories=5`: Keep only top 5 categories
- `mode='REPLACE'`: Modify original field
- `use_vectorization=False`: Simple processing for small data
- `generate_visualization=True`: Create charts
- `save_output=True`: Save to files
- `force_recalculation=False`: Use cache if available

In [ ]:
# Create operation with production-style configuration
operation = CategoricalGeneralizationOperation(
    # Core parameters
    field_name=field_name,               # From config
    mode='REPLACE',
    strategy='frequency_based',
    
    # Output configuration
    column_prefix='_',
    output_field_name=None,
    output_format='csv',
    
    # Frequency-based strategy parameters
    freq_threshold=0.1,              # Group categories < 10%
    max_categories=5,                # Keep max 5 categories
    min_group_size=2,                # Minimum records per group
    
    # Rare value handling
    group_rare_as='OTHER',
    rare_value_template='OTHER',
    
    # Text processing
    case_sensitive=False,
    text_normalization='basic',
    fuzzy_matching=False,
    similarity_threshold=0.85,
    
    # Privacy settings
    privacy_check_enabled=True,
    min_acceptable_k=2,
    max_acceptable_disclosure_risk=0.3,
    quasi_identifiers=None,
    ka_risk_field=None,
    risk_threshold=5,
    vulnerable_record_strategy='generalize',
    
    # Processing settings
    use_vectorization=False,         # Disable for small data
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,     # Create visualization artifacts
    save_output=True,                # Save results to files
    force_recalculation=False        # Use cache when available
)

print("✓ Operation configured")
print(f"  Field:            {operation.field_name}")
print(f"  Strategy:         {operation.strategy}")
print(f"  Threshold:        {operation.freq_threshold:.0%}")
print(f"  Max categories:   {operation.max_categories}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing categorical generalization...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Load the anonymized output from task_dir
- Compare with original data
- Review metrics and artifacts

**Generated artifacts:**
- Anonymized CSV in output/
- Metrics JSON in metrics/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records
    print("\n🔍 Sample Anonymized Records:")
    display_cols = [field_name]
    if 'record_id' in result_df.columns:
        display_cols.append('record_id')
    if 'region' in result_df.columns:
        display_cols.append('region')
    if 'value' in result_df.columns:
        display_cols.append('value')
    print(result_df[display_cols].head(10))
    
    # Distribution comparison
    print(f"\n📈 {field_name.title()} Distribution (After Anonymization):")
    dist = result_df[field_name].value_counts()
    for cat, count in dist.items():
        pct = (count / len(result_df)) * 100
        bar = '█' * int(pct / 5)
        print(f"  {cat:30s} {bar:20s} {count:2d} ({pct:5.1f}%)")
    
    # Calculate metrics
    original_unique = df[field_name].nunique()
    generalized_unique = result_df[field_name].nunique()
    reduction_pct = (original_unique - generalized_unique) / original_unique * 100
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original {field_name}s:    {original_unique}")
    print(f"  Generalized {field_name}s: {generalized_unique}")
    print(f"  Reduction:              {reduction_pct:.1f}%")
    
    # Display result metrics
    if result.metrics:
        print("\n🔒 Privacy Metrics:")
        for key, value in list(result.metrics.items())[:10]:  # Show first 10
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Lower {field_name}s = Better privacy protection!")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Anonymized CSV
├── metrics/          # JSON metrics
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

# List all subdirectories and files
for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:  # Show first 5 files
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")
        if len(files) > 5:
            print(f"   ... and {len(files) - 5} more files")
        print()

print("=" * 80)
print("\n✅ All artifacts saved successfully!")
print(f"\n💡 TIP: Navigate to {task_dir.absolute()} to inspect files manually")

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. Display Metrics JSON (NEWEST FILE)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # Sort by modification time (newest first)
        metrics_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = metrics_files[0]
        
        print(f"✓ Found {len(metrics_files)} metrics file(s)")
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        
        # Show file info
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # Display metadata
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if 'operation' in metadata:
                op = metadata['operation']
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
            
            # Display effectiveness metrics
            if 'effectiveness' in metrics:
                print("\n📈 EFFECTIVENESS:")
                eff = metrics['effectiveness']
                print(f"   Total Records: {eff.get('total_records', 'N/A'):,}")
                print(f"   Original Unique: {eff.get('original_unique', 'N/A')}")
                print(f"   Anonymized Unique: {eff.get('anonymized_unique', 'N/A')}")
                print(f"   Effectiveness Ratio: {eff.get('effectiveness_ratio', 0):.4f}")
                print(f"   Null Increase: {eff.get('null_increase', 0):.4f}")
            
            # Display performance metrics
            if 'performance' in metrics:
                print("\n⚡ PERFORMANCE:")
                perf = metrics['performance']
                print(f"   Duration: {perf.get('duration_seconds', 0):.4f}s")
                print(f"   Records Processed: {perf.get('records_processed', 0):,}")
                print(f"   Records/Second: {perf.get('records_per_second', 0):.2f}")
            
            # Display field info (original vs processed)
            if 'field_info' in metrics:
                print("\n📊 FIELD COMPARISON:")
                field_info = metrics['field_info']
                
                # Original
                if 'original' in field_info:
                    orig = field_info['original']
                    print(f"\n   ORIGINAL:")
                    print(f"      Total Count: {orig.get('total_count', 0):,}")
                    print(f"      Unique Count: {orig.get('unique_count', 0)}")
                    print(f"      Uniqueness Ratio: {orig.get('uniqueness_ratio', 0):.2%}")
                    print(f"      Null Count: {orig.get('null_count', 0)}")
                    
                    if 'top_values' in orig and orig['top_values']:
                        print(f"\n      Top Values:")
                        for val, count in list(orig['top_values'].items())[:5]:
                            print(f"         {val:<30}: {count:>3}")
                
                # Processed
                if 'processed' in field_info:
                    proc = field_info['processed']
                    print(f"\n   PROCESSED:")
                    print(f"      Total Count: {proc.get('total_count', 0):,}")
                    print(f"      Unique Count: {proc.get('unique_count', 0)}")
                    print(f"      Uniqueness Ratio: {proc.get('uniqueness_ratio', 0):.2%}")
                    print(f"      Null Count: {proc.get('null_count', 0)}")
                    
                    if 'top_values' in proc and proc['top_values']:
                        print(f"\n      Top Values:")
                        for val, count in list(proc['top_values'].items())[:5]:
                            print(f"         {val:<30}: {count:>3}")
            
            # Display generalization info
            if 'generalization' in metrics:
                print("\n🔄 GENERALIZATION:")
                gen = metrics['generalization']
                print(f"   Strategy: {gen.get('strategy', 'N/A')}")
                print(f"   Reduction Ratio: {gen.get('reduction_ratio', 0):.2%}")
                
                if 'parameters' in gen:
                    params = gen['parameters']
                    print(f"\n   Parameters:")
                    print(f"      Max Categories: {params.get('max_categories', 'N/A')}")
                    print(f"      Min Group Size: {params.get('min_group_size', 'N/A')}")
                    print(f"      Group Rare As: {params.get('group_rare_as', 'N/A')}")
                    print(f"      Case Sensitive: {params.get('case_sensitive', 'N/A')}")
            
            # Display information loss metrics
            if 'categorical_info_loss' in metrics:
                print("\n📉 INFORMATION LOSS:")
                info_loss = metrics['categorical_info_loss']
                print(f"   Precision Loss: {info_loss.get('precision_loss', 0):.2%}")
                print(f"   Entropy Loss: {info_loss.get('entropy_loss', 0):.2%}")
                print(f"   Distribution Shift: {info_loss.get('distribution_shift', 0):.2%}")
                print(f"   Category Reduction: {info_loss.get('category_reduction_ratio', 0):.2%}")
                print(f"   Avg Group Size: {info_loss.get('average_group_size', 0):.2f}")
            
            # Display distribution metrics
            if 'distribution_metrics' in metrics:
                print("\n📊 DISTRIBUTION METRICS:")
                dist = metrics['distribution_metrics']
                
                if 'original' in dist:
                    orig = dist['original']
                    print(f"\n   Original:")
                    print(f"      Unique Values: {orig.get('unique_values', 0)}")
                    print(f"      Entropy: {orig.get('entropy', 0):.4f}")
                
                if 'anonymized' in dist:
                    anon = dist['anonymized']
                    print(f"\n   Anonymized:")
                    print(f"      Unique Values: {anon.get('unique_values', 0)}")
                    print(f"      Entropy: {anon.get('entropy', 0):.4f}")
            
            # Display privacy metrics
            if 'privacy_metrics' in metrics:
                print("\n🔒 PRIVACY METRICS:")
                privacy = metrics['privacy_metrics']
                status = privacy.get('status', 'N/A')
                print(f"   Status: {status}")
                if status == 'SKIPPED':
                    print(f"   Reason: {privacy.get('reason', 'N/A')}")
                print(f"   Summary: {metrics.get('privacy_summary', 'N/A')}")
            
            # Display other key metrics
            print("\n📌 OTHER METRICS:")
            other_metrics = [
                ('semantic_diversity', 'Semantic Diversity'),
                ('processing_rate', 'Processing Rate'),
                ('chunk_size_used', 'Chunk Size Used'),
                ('adaptive_chunk_size', 'Adaptive Chunk Size')
            ]
            
            for key, label in other_metrics:
                if key in metrics:
                    value = metrics[key]
                    if isinstance(value, float):
                        if value < 1:
                            print(f"   {label}: {value:.2%}")
                        else:
                            print(f"   {label}: {value:.4f}")
                    elif isinstance(value, bool):
                        print(f"   {label}: {'Yes' if value else 'No'}")
                    else:
                        print(f"   {label}: {value}")
        
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. Display Output Data Sample (NEWEST FILE)
print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        # Sort by modification time (newest first)
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        # Show file info
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            print(f"📊 Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")
            print(f"\n{output_df.head(10).to_string()}")
            
            # Show value counts for the anonymized field
            if field_name in output_df.columns:
                print(f"\n\n📊 {field_name.title()} Distribution:")
                print("-" * 80)
                dist = output_df[field_name].value_counts()
                total = len(output_df)
                print(f"{'Category':<40} {'Count':>6} {'Percentage':>12}")
                print("-" * 60)
                for cat, count in dist.items():
                    pct = (count / total) * 100
                    bar = '█' * int(pct / 2)
                    print(f"{str(cat):<40} {count:>6} {pct:>11.2f}%")
            else:
                print(f"\n⚠️  Field '{field_name}' not found in output")
                print(f"Available columns: {', '.join(output_df.columns)}")
        
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. Display Visualizations (NEWEST FILES ONLY - matching latest operation)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                name_parts = viz_file.stem.split('_')
                # Remove field name prefix and timestamp suffix
                if len(name_parts) > 3:
                    viz_name = ' '.join(name_parts[2:-1]).replace('categorical generalization', '').strip().title()
                else:
                    viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

# 4. Display Category Mapping (NEWEST FILE)
print("\n\n4️⃣ CATEGORY MAPPING:")
print("-" * 80)

# Search for mapping files in all subdirectories
mapping_files = []
for pattern in ['*mapping*.json', '*_mapping_*.json']:
    mapping_files.extend(task_dir.glob(f'**/{pattern}'))

# Remove duplicates
mapping_files = list(set(mapping_files))

if not mapping_files:
    print("   ℹ️  No mapping file found")
else:
    # Sort by modification time (newest first)
    mapping_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    latest_mapping_file = mapping_files[0]
    
    print(f"✓ Found {len(mapping_files)} mapping file(s)")
    print(f"📄 Loading LATEST: {latest_mapping_file.name}")
    
    # Show file info
    mtime = datetime.fromtimestamp(latest_mapping_file.stat().st_mtime)
    print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Size: {latest_mapping_file.stat().st_size / 1024:.1f} KB\n")
    
    try:
        with open(latest_mapping_file, 'r', encoding='utf-8') as f:
            mapping_data = json.load(f)
        
        # Display mapping metadata
        print(f"📋 Mapping Info:")
        print(f"   Operation ID: {mapping_data.get('operation_id', 'N/A')}")
        print(f"   Field Name: {mapping_data.get('field_name', 'N/A')}")
        print(f"   Strategy: {mapping_data.get('strategy', 'N/A')}")
        print(f"   Timestamp: {mapping_data.get('timestamp', 'N/A')}")
        print(f"   Total Mappings: {mapping_data.get('total_mappings', 0)}")
        
        # Display mappings
        if 'mappings' in mapping_data:
            mappings = mapping_data['mappings']
            
            print(f"\n📋 Category Mappings:")
            print("-" * 60)
            print(f"{'Original':<40} → {'Generalized':<30}")
            print("-" * 60)
            
            # Sort mappings alphabetically
            sorted_mappings = sorted(mappings.items())
            
            # Show all mappings (grouped by result)
            unchanged = []
            changed = []
            
            for original, generalized in sorted_mappings:
                if original == generalized:
                    unchanged.append((original, generalized))
                else:
                    changed.append((original, generalized))
            
            # Show changed first (more interesting)
            if changed:
                print("\n  📝 Changed Categories:")
                for original, generalized in changed:
                    print(f"     {original:<37} → {generalized}")
            
            if unchanged:
                print(f"\n  ✓ Unchanged Categories ({len(unchanged)}):")
                for original, generalized in unchanged[:5]:
                    print(f"     {original:<37} → {generalized}")
                if len(unchanged) > 5:
                    print(f"     ... and {len(unchanged) - 5} more unchanged")
            
            # Summary
            print(f"\n📊 Summary:")
            print(f"   Total Mappings: {len(mappings)}")
            print(f"   Changed: {len(changed)}")
            print(f"   Unchanged: {len(unchanged)}")
            if len(mappings) > 0:
                print(f"   Reduction: {len(changed) / len(mappings) * 100:.1f}% of categories generalized")
        else:
            print(f"\n⚠️  No 'mappings' key found in file")
    
    except json.JSONDecodeError as e:
        print(f"❌ JSON decode error: {e}")
    except Exception as e:
        print(f"❌ Error reading mapping: {e}")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ **Load data**: Read CSV from examples/data_examples/  
✅ **Setup environment**: TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Full production-style parameters with field config 
✅ **Execute**: Use operation.execute() with explicit configs
✅ **Analyze results**: Review metrics and artifacts

**Key takeaways:**
- `operation.execute()` handles end-to-end processing
- Execution behavior (visualizations, output saving, caching) configured in operation constructor
- Field name is configurable via `create_config_kwargs()`
- Frequency-based strategy groups rare categories
- Fewer categories = better privacy
- All artifacts saved in structured directories

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_categorical_generalization_advanced.ipynb`** includes:
- All 3 strategies (hierarchy, frequency_based, merge_low_freq)
- Custom hierarchy dictionaries
- Multi-stage processing pipelines
- Advanced privacy metrics (k-anonymity, l-diversity)
- Performance optimization
- Production deployment patterns

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🎓 [Privacy Metrics Guide](../../docs/privacy_metrics.md)